There is a bug in `pytorch` and you need to make sure that in the conda environment the following variable is set:
```
conda env config vars set KMP_DUPLICATE_LIB_OK=TRUE
```
You can check that the variable is set by running:
```
conda env config vars list
```

In [ ]:
from collections import defaultdict

import torch
import torchrl
import tensordict

from envs.mh5robotenv import MH5RobotEnv

from tensordict.nn import  TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor

from torch import nn

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import GymEnv, TransformedEnv, Compose, ObservationNorm, DoubleToFloat, StepCounter
from torchrl.envs.utils import check_env_specs, set_exploration_type, ExplorationType

from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

from tqdm import tqdm


In [ ]:
print(f"torch version: {torch.__version__}")
print(f"torchrl version: {torchrl.__version__}")
print(f"tensordict version: {tensordict.__version__}")

In [ ]:
device = torch.device("cpu")
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
# if torch.cuda.is_available():
#     device = torch.device("cuda")
print(device)

In [ ]:
base_env = GymEnv("MH5Robot-v8", device=device)

In [ ]:
env = TransformedEnv(
    base_env,
    Compose(
        ObservationNorm(in_keys=['observation']),
        DoubleToFloat(),
        StepCounter(),
    ),
)

In [ ]:
env.transform[0].init_stats(num_iter=1000, reduce_dim=0, cat_dim=0)
print("normalization constant shape:", env.transform[0].loc.shape)

In [ ]:
print("observation_spec:", env.observation_spec)
print("reward_spec:", env.reward_spec)
print("input_spec:", env.input_spec)
print("action_spec (as defined by input_spec):", env.action_spec)

In [ ]:
check_env_specs(env)

In [ ]:
rollout = env.rollout(3)
print("rollout of three steps:", rollout)
print("Shape of the rollout TensorDict:", rollout.batch_size)

In [ ]:
config = {
    'num_cells': 256,
    'frames_per_batch': 10_000,
    'total_frames': 2_000_000,
    'gamma': 0.99,
    'lmbda': 0.95,
    'clip_epsilon': 0.2,
    'entropy_eps': 1e-4,
    'lr': 3e-4,
    'num_epochs': 10,
    'sub_batch_size': 64,
    'max_grad_norm': 1.0,
}

In [ ]:
actor_net = nn.Sequential(
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(2 * env.action_spec.shape[-1], device=device),
    NormalParamExtractor(),
)

policy_module = TensorDictModule(actor_net, in_keys=['observation'], out_keys=['loc', 'scale'])

policy_module = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec,
    in_keys=['loc', 'scale'],
    distribution_class=TanhNormal,
    distribution_kwargs={
        'low': env.action_spec_unbatched.space.low,
        'high': env.action_spec_unbatched.space.high,
    },
    return_log_prob=True,
)

In [ ]:
value_net = nn.Sequential(
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(1, device=device)
)

value_module = ValueOperator(
    module=value_net,
    in_keys=['observation'],
)

In [ ]:
# we need to do this to "initialize" the Lazy modules
print("Running policy:", policy_module(env.reset()))
print("Running value:", value_module(env.reset()))

In [ ]:
collector = SyncDataCollector(
    create_env_fn=env,
    policy=policy_module,
    frames_per_batch=config['frames_per_batch'],
    total_frames=config['total_frames'],
    split_trajs=False,
    device=device,
    use_buffers=False,  # https://github.com/pytorch/rl/issues/3066#issuecomment-3077398138
)

In [ ]:
replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=config['frames_per_batch']),
    sampler=SamplerWithoutReplacement(),
)

In [ ]:
advantage_module = GAE(
    gamma=config['gamma'],
    lmbda=config['lmbda'],
    value_network=value_module,
    average_gae=True,
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=config['clip_epsilon'],
    entropy_bonus=bool(config['entropy_eps']),
    entropy_coeff=config['entropy_eps'],
    critic_coeff=1.0,
    loss_critic_type='smooth_l1',
)

optim = torch.optim.Adam(loss_module.parameters(), config['lr'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optim,
    T_max=(config['total_frames'] // config['frames_per_batch']),
    eta_min=0.0
)

In [ ]:
logs = defaultdict(list)
pbar = tqdm(total=config['total_frames'])
eval_str = ""

# collector.verbose = True

for i, tensordict_data in enumerate(collector):
    for _ in range (config['num_epochs']):
        advantage_module(tensordict_data)
        data_view = tensordict_data.reshape(-1)
        replay_buffer.extend(data_view.cpu())
        for _ in range(config['frames_per_batch'] // config['sub_batch_size']):
            subdata = replay_buffer.sample(config['sub_batch_size'])
            loss_vals = loss_module(subdata.to(device))
            loss_value = (
                loss_vals['loss_objective']
                + loss_vals['loss_critic']
                + loss_vals['loss_entropy']
            )
            loss_value.backward()
            torch.nn.utils.clip_grad_norm_(loss_module.parameters(), config['max_grad_norm'])
            optim.step()
            optim.zero_grad()

    logs['reward'].append(tensordict_data['next', 'reward'].mean().item())
    pbar.update(tensordict_data.numel())
    cum_reward_str = f"average reward={logs['reward'][-1]:4.4f} (init={logs['reward'][0]:4.4f})"
    logs['step_count'].append(tensordict_data['step_count'].max().item())
    stepcount_str = f"step count (max): {logs['step_count'][-1]}"
    logs['lr'].append(optim.param_groups[0]['lr'])
    lr_str = f"lr policy: {logs['lr'][-1]:4.4f}"

    if i % 10 == 0:
        with set_exploration_type(ExplorationType.DETERMINISTIC), torch.no_grad():
            eval_rollout = env.rollout(1000, policy_module)
            logs['eval reward'].append(eval_rollout['next', 'reward'].mean().item())
            logs["eval reward (sum)"].append(eval_rollout["next", "reward"].sum().item())
            logs["eval step_count"].append(eval_rollout["step_count"].max().item())
            eval_str = (
                f"eval cummulative reward: {logs['eval reward (sum)'][-1]:4.4f} "
                f"(init: {logs['eval reward (sum)'][0]:4.4f})"
                f"eval step-count: {logs['eval step_count'][-1]}"
            )
            del eval_rollout

    pbar.set_description(", ".join([eval_str, cum_reward_str, stepcount_str, lr_str]))

    scheduler.step()


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(logs["reward"])
plt.title("training rewards (average)")
plt.subplot(2, 2, 2)
plt.plot(logs["step_count"])
plt.title("Max step count (training)")
plt.subplot(2, 2, 3)
plt.plot(logs["eval reward (sum)"])
plt.title("Return (test)")
plt.subplot(2, 2, 4)
plt.plot(logs["eval step_count"])
plt.title("Max step count (test)")
plt.show()